# Compare custom model to LDA baseline

This example shows how to:
1. Run a custom model and save results
2. Compare to LDA baseline with corrected t-test

Prerequisites:
- Run LDA baseline first: python scripts/classify_dataset.py -m lda --seeds 42 43 44 45 46
  (creates `all_cboth_mlda_s42-43-44-45-46.json` in `paths.results_cache`)

### Imports

In [1]:
import json
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from fnirsatlas.feature_extraction import WindowedFeatureExtractor
from fnirsatlas.classification import run_benchmark
from fnirsatlas.compare import compare_models, format_comparison_table
from fnirsatlas.utils import format_seeds

### Define custom model

Create a 3-step pipeline:
1. **WindowedFeatureExtractor**: Extracts standard deviation from each channel
2. **StandardScaler**: Z-normalizes features across samples
3. **SVC**: Support Vector Classifier with RBF kernel

In [2]:
def get_custom_svm(**kwargs):
    """Custom SVM with RBF kernel model."""
    random_state = kwargs["random_state"]
    return Pipeline(
        [
            (
                "feature_extractor",
                WindowedFeatureExtractor(n_windows=1, features=["std"]),
            ),
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(kernel="rbf", random_state=random_state, C=1.0, gamma="scale"),
            ),
        ]
    )

### Run the benchmark with the custom model

Execute cross-subject and within-subject classification on the MA suite with hyperparameter tuning using nested cross-validation (5 × #seeds outer CV, 4-fold inner CV by default).

In [3]:
seeds = [42, 43, 44, 45, 46]
classification_setting = "both"

print("Running custom SVM with RBF kernel model...")
custom_results = run_benchmark(
    get_clf=get_custom_svm,
    suite="ma",
    classification_setting="both",
    seeds=seeds,
    n_jobs=1,
    param_grid={
        "clf__C": [1e-3, 1, 1e3],
    },
)
custom_results["model_name"] = "Custom SVM RBF"


Running custom SVM with RBF kernel model...


Classifying datasets: 100%|██████████| 4/4 [00:58<00:00, 14.67s/it, shin2019__ma]


Summary:
dataset          task                           #classes  #epochs   Cross%    Within%   Cross κ    Within κ
--------------------------------------------------------------------------------------------------------------
Bnci             BL|MA                                 2      348  48.0±5.1   59.4±12.1  -3.9±10.2    20.0±22.2
Shin2016 (ma)    BL|MA                                 2     1740  51.2±2.7    58.7±9.1    2.3±5.4    17.5±18.2
Shin2018b        BL|MA                                 2      720  56.5±4.5   62.3±11.2   13.0±9.0    24.7±22.5
Shin2019 (ma)    BL|MA                                 2     1080  49.7±3.8   55.5±10.6   -0.6±7.7    11.1±21.3
--------------------------------------------------------------------------------------------------------------
MEAN                                                               51.4±4.0   59.0±10.8    2.7±8.1    18.3±21.0


### Load baseline results

Load the pre-computed LDA baseline results for comparison.

If you haven't run the LDA baseline yet, run first using `python scripts/classify_dataset.py -m lda --seeds 42 43 44 45 46`.

In [4]:

# Check if LDA baseline exists
lda_file = (
    Path("results")
    / f"all_c{classification_setting}_mlda_s{format_seeds(seeds)}.json"
)
if not lda_file.exists():
    raise FileNotFoundError(
        "LDA baseline not found!\nPlease run first:\n"
        "    python scripts/classify_dataset.py -m lda --seeds 42 43 44 45 46"
    )

# Load LDA baseline results
with open(lda_file, "r") as f:
    lda_results = json.load(f)
lda_results["model_name"] = "LDA Baseline"

### Compare custom model to LDA baseline

Perform statistical comparison using corrected t-test and FDR correction.

The comparison uses:
- **Corrected t-test** (Nadeau & Bengio, 2003) for cross-validation dependency
- **FDR correction** (Benjamini-Hochberg) for multiple comparisons  
- **Significance** based on FDR-corrected p-values

In [5]:
# Compare to LDA baseline
comparisons = compare_models(
    [custom_results, lda_results], metric="cross_acc", alpha=0.05
)
format_comparison_table(comparisons, alpha=0.05, metric="cross_acc")

# Save custom model results
custom_file = Path("results") / f"ma_c{classification_setting}_custom_svm_s{format_seeds(seeds)}.json"
with open(custom_file, "w") as f:
    json.dump(custom_results, f, indent=2)
    
print(f"\nResults files saved:")
print(f"  Custom SVM: {custom_file}")
print(f"  LDA Baseline: {lda_file}")
print(f"\nTo re-run comparison later:")
print(f"  python scripts/compare_models.py {custom_file} {lda_file}")


Custom SVM RBF vs LDA Baseline cross_acc comparison (α = 0.05)
Dataset          Task                                 CUSTOM          LDA         Diff   P-Orig    P-FDR Sig
------------------------------------------------------------------------------------------------------------------------
Shin2016 (ma)    BL|MA                              51.2±2.7     69.1±3.5    -17.9±4.4    0.000    0.000 ***
Shin2018b        BL|MA                              56.5±4.5     69.9±4.8    -13.4±3.9    0.000    0.000 ***
Bnci             BL|MA                              48.0±5.1    78.1±10.9   -30.1±10.0    0.000    0.000 ***
Shin2019 (ma)    BL|MA                              49.7±3.8     65.8±5.1    -16.2±7.0    0.000    0.000 ***

Results files saved:
  Custom SVM: results/ma_cboth_custom_svm_s42-43-44-45-46.json
  LDA Baseline: results/all_cboth_mlda_s42-43-44-45-46.json

To re-run comparison later:
  python scripts/compare_models.py results/ma_cboth_custom_svm_s42-43-44-45-46.json results/all_

### Results Interpretation

Understanding the comparison output:
- The comparison table shows performance differences between Custom SVM RBF and LDA Baseline
- "Diff" column shows the accuracy difference (Custom - LDA)
- "P-Orig" shows uncorrected p-values from paired t-tests
- "P-FDR" shows False Discovery Rate corrected p-values (Benjamini-Hochberg)
- "Sig" shows significance levels: *** (p < 0.001), ** (p < 0.01), * (p < 0.05)
- Negative differences indicate LDA performs better than Custom SVM
- The results show that the LDA baseline significantly outperforms the Custom SVM across all datasets